In [1]:
print("Machine Learning CNN Google Colab + Python + TensorFlow/Keras")

Machine Learning CNN Google Colab + Python + TensorFlow/Keras


In [2]:
print("Phase 1 alpha: simple classification using the datset Mendeley Data – Kermany et al. ")

Phase 1 alpha: simple classification using the datset Mendeley Data – Kermany et al. 


The main task for this one project is that to ensure that we have to implement the folllowing tasks :
1. first test the model at the avalibale data set from the interent
2. if the model is able to classify based on the trained dataset as it a Normal,Viral,Bacterial /
3. the data set is having a good number of expected test cases for the multivariate classifictaion ?

In [ ]:
MyDrive/Pneumonia_Project/Dataset/

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:

# importing the path and all the releated thing for the dataset classification
from pathlib import Path
PROJECT_DIR = Path('/content/drive/MyDrive/Pneumonia_Project')
ZIP_PATH = PROJECT_DIR / 'Dataset' / 'ChestXRay2017.zip'
EXTRACT_DIR = PROJECT_DIR / 'Dataset' / 'Extracted'
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
if ZIP_PATH.exists():

    print("Dataset ZIP file found!")
    import zipfile

    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)

    print("Dataset extraction completed!")

else:
    print("ERROR: Dataset ZIP file not found.")
    print("Please upload ChestXRay2017.zip to Google Drive.")


In [ ]:
#extracting all the required libraries for the CNN checking avaloiablitiy of each and every version imported
import os
import random
import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score
)

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("OpenCV:", cv2.__version__)
print("TensorFlow:", tf.__version__)
print("Available GPU:", tf.config.list_physical_devices('GPU'))


In [ ]:

# Display extracted dataset folders

for root, dirs, files in os.walk(EXTRACT_DIR):
    print("Folder:", root)
    print("Subfolders:", dirs[:5])
    print("Number of files:", len(files))
    print("-" * 40)

    # Display only the first two directory levels
    if root.count(os.sep) - str(EXTRACT_DIR).count(os.sep) >= 2:
        dirs[:] = []


In [ ]:
# normal and phemumonia detection

candidates = []

for path in DATA_DIR.rglob("*"):

    if path.is_dir():

        folder_names = {
            p.name.upper()
            for p in path.iterdir()
            if p.is_dir()
        }

        if {"NORMAL", "PNEUMONIA"}.issubset(folder_names):
            candidates.append(path)

if len(candidates) == 0:
    raise FileNotFoundError(
        "Could not locate NORMAL and PNEUMONIA folders."
    )

# Select the common chest_xray directory
CHEST_DIR = candidates[0]

print("CHEST X-RAY DIRECTORY:")
print(CHEST_DIR)

print("\nSUBDIRECTORIES:")

for folder in CHEST_DIR.iterdir():
    if folder.is_dir():
        print(folder.name)


In [ ]:
#extracting the features
def identify_label(image_path):

    filename = image_path.name.lower()
    parent = image_path.parent.name.upper()

    if parent == "NORMAL":
        return "Normal"

    if "bacteria" in filename:
        return "Bacterial"

    if "virus" in filename:
        return "Viral"

    return None


# image paths
image_extensions = {
    ".jpg", ".jpeg", ".png"
}

image_records = []

for image_path in CHEST_DIR.rglob("*"):

    if image_path.suffix.lower() not in image_extensions:
        continue

    label = identify_label(image_path)

    image_records.append({
        "image_path": str(image_path),
        "filename": image_path.name,
        "label": label
    })


df = pd.DataFrame(image_records)

print("TOTAL IMAGE FILES:", len(df))

print("\nLABEL DISTRIBUTION:")

print(
    df["label"]
    .value_counts(dropna=False)
)

print("\nUNIDENTIFIED FILES:")

print(
    df[df["label"].isna()]
    .head(10)
)

# Stop if any pneumonia subtype cannot be identified
if df["label"].isna().any():
    raise ValueError(
        "Some images have unidentified labels. "
        "Inspect the filenames before proceeding."
    )


In [ ]:

def inspect_image(path):
  try:
        with Image.open(path) as img:

            img.verify()


        with Image.open(path) as img:

            width, height = img.size
            mode = img.mode


        with open(path, "rb") as file:
            image_hash = hashlib.md5(
                file.read()
            ).hexdigest()

        return {
            "valid": True,
            "width": width,
            "height": height,
            "mode": mode,
            "hash": image_hash
        }

    except Exception:
        return {
            "valid": False,
            "width": None,
            "height": None,
            "mode": None,
            "hash": None
        }

inspection = [
    inspect_image(path)
    for path in df["image_path"]
]

inspection_df = pd.DataFrame(inspection)

df = pd.concat(
    [df.reset_index(drop=True),
     inspection_df],
    axis=1
)

print("IMAGE QUALITY REPORT")
print("=" * 45)

print("Total images:", len(df))

print("Valid images:", df["valid"].sum())

print(
    "Corrupted images:",
    (~df["valid"]).sum()
)
df = df[df["valid"]].copy()
duplicate_hashes = df[
    df.duplicated("hash", keep=False)
]

print("\nDUPLICATE IMAGE REPORT")

print(
    "Images involved in duplicate groups:",
    len(duplicate_hashes)
)
conflicting_hashes = (
    duplicate_hashes.groupby("hash")["label"]
    .nunique()
)

conflicting_hashes = conflicting_hashes[
    conflicting_hashes > 1
].index

print(
    "Conflicting-label duplicate groups:",
    len(conflicting_hashes)
)
df = df[
    ~df["hash"].isin(conflicting_hashes)
].copy()
df = df.drop_duplicates(
    subset="hash",
    keep="first"
).reset_index(drop=True)

print("\nCLEANED DATASET")

print("Remaining images:", len(df))

print("\nClass counts:")
print(df["label"].value_counts())


In [ ]:
class_counts = (
    df["label"]
    .value_counts()
    .reindex(CLASS_NAMES)
)

plt.figure(figsize=(9, 5))

sns.barplot(
    x=class_counts.index,
    y=class_counts.values
)

plt.title("Chest X-Ray Dataset Distribution")

plt.xlabel("Classification Category")

plt.ylabel("Number of Images")

plt.grid(axis="y", alpha=0.3)

plt.tight_layout()

plt.show()
print("DATASET DISTRIBUTION")

for label, count in class_counts.items():

    percentage = (
        count / len(df)
    ) * 100

    print(
        f"{label}: {count} images "
        f"({percentage:.2f}%)"
    )


In [ ]:
plt.figure(figsize=(12, 8))

for index, label in enumerate(CLASS_NAMES):

    samples = df[
        df["label"] == label
    ].sample(
        min(3, (df["label"] == label).sum()),
        random_state=SEED
    )

    for j, (_, row) in enumerate(samples.iterrows()):

        image = Image.open(
            row["image_path"]
        ).convert("L")

        plt.subplot(3, 3, index * 3 + j + 1)

        plt.imshow(image, cmap="gray")

        plt.title(label)

        plt.axis("off")

plt.suptitle(
    "Sample Chest X-Ray Images",
    fontsize=16
)

plt.tight_layout()

plt.show()


In [ ]:
def extract_patient_id(row):

    filename = row["filename"].lower()
    label = row["label"]

    if label in ["Bacterial", "Viral"]:

        match = re.search(
            r"person(\d+)",
            filename
        )

        if match:
            return (
                label.lower()
                + "_person_"
                + match.group(1)
            )

    if label == "Normal":

        match = re.search(
            r"im-(\d+)",
            filename
        )

        if match:
            return "normal_im_" + match.group(1)

    return "unparsed_" + row["hash"]


df["patient_group"] = df.apply(
    extract_patient_id,
    axis=1
)

print("PATIENT GROUP ANALYSIS")

print(
    "Unique groups:",
    df["patient_group"].nunique()
)

print(
    "Images with unparsed patient IDs:",
    df["patient_group"]
    .str.startswith("unparsed_")
    .sum()
)

print("\nImages per class and patient group:")

print(
    df.groupby("label")["patient_group"]
    .nunique()
)


In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
label_mapping = {
    "Normal": 0,
    "Bacterial": 1,
    "Viral": 2
}

df["label_id"] = df["label"].map(
    label_mapping
)
outer_split = StratifiedGroupKFold(
    n_splits=7,
    shuffle=True,
    random_state=SEED
)

train_val_indices, test_indices = next(
    outer_split.split(
        df,
        df["label_id"],
        groups=df["patient_group"]
    )
)

train_val_df = df.iloc[
    train_val_indices
].copy()

test_df = df.iloc[
    test_indices
].copy()
inner_split = StratifiedGroupKFold(
    n_splits=6,
    shuffle=True,
    random_state=SEED
)

train_indices, val_indices = next(
    inner_split.split(
        train_val_df,
        train_val_df["label_id"],
        groups=train_val_df["patient_group"]
    )
)

train_df = train_val_df.iloc[
    train_indices
].copy()

val_df = train_val_df.iloc[
    val_indices
].copy()
train_groups = set(train_df["patient_group"])
val_groups = set(val_df["patient_group"])
test_groups = set(test_df["patient_group"])

assert train_groups.isdisjoint(val_groups)
assert train_groups.isdisjoint(test_groups)
assert val_groups.isdisjoint(test_groups)
def display_split(name, data):

    print("\n", name)

    print("-" * 45)

    print("Images:", len(data))

    print("Unique groups:", data["patient_group"].nunique())

    print(
        data["label"]
        .value_counts()
        .reindex(CLASS_NAMES, fill_value=0)
    )


display_split("TRAINING DATA", train_df)

display_split("VALIDATION DATA", val_df)

display_split("TESTING DATA", test_df)

print("\nPATIENT GROUP OVERLAP CHECK: PASSED")


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def preprocess_image(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.io.decode_image(
        image,
        channels=1,
        expand_animations=False
    )
    image.set_shape([None, None, 1])
    image = tf.image.resize(
        image,
        [IMG_SIZE, IMG_SIZE]
    )
    image = image / 127.5 - 1.0

    return image, label


def create_dataset(dataframe, training=False):

    dataset = tf.data.Dataset.from_tensor_slices(
        (
            dataframe["image_path"].values,
            dataframe["label_id"].values
        )
    )

    dataset = dataset.map(
        preprocess_image,
        num_parallel_calls=AUTOTUNE
    )

    if training:

        dataset = dataset.shuffle(
            buffer_size=len(dataframe),
            seed=SEED
        )

    dataset = dataset.batch(BATCH_SIZE)

    dataset = dataset.prefetch(AUTOTUNE)

    return dataset


train_ds = create_dataset(
    train_df,
    training=True
)

val_ds = create_dataset(
    val_df
)

test_ds = create_dataset(
    test_df
)

print("DATA PIPELINES CREATED")

print("Training batches:", tf.data.experimental.cardinality(train_ds).numpy())

print("Validation batches:", tf.data.experimental.cardinality(val_ds).numpy())

print("Testing batches:", tf.data.experimental.cardinality(test_ds).numpy())


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1, 2]),
    y=train_df["label_id"].values
)

class_weights = {
    i: float(weight)
    for i, weight in enumerate(class_weights_array)
}

print("CLASS WEIGHTS")

for i, weight in class_weights.items():

    print(
        CLASS_NAMES[i],
        ":", round(weight, 4)
    )

print("\nTRAINING CLASS COUNTS")

print(
    train_df["label"]
    .value_counts()
    .reindex(CLASS_NAMES, fill_value=0)
)


In [ ]:
from tensorflow.keras import layers, models
augmentation = models.Sequential(
    [
        layers.RandomRotation(
            0.02,
            fill_mode="reflect"
        ),

        layers.RandomTranslation(
            0.02,
            0.02,
            fill_mode="reflect"
        )
    ],
    name="data_augmentation"
)
inputs = layers.Input(
    shape=(IMG_SIZE, IMG_SIZE, 1)
)

x = augmentation(inputs)
blocks = [
    (32, "block1"),
    (64, "block2"),
    (128, "block3"),
    (256, "block4"),
    (256, "block5")
]


for filters, block_name in blocks:
    x = layers.Conv2D(
        filters,
        (3, 3),
        padding="same",
        activation="tanh",
        name=f"{block_name}_conv1"
    )(x)
    x = layers.Conv2D(
        filters,
        (3, 3),
        padding="same",
        activation="tanh",
        name=f"{block_name}_conv2"
    )(x)
    x = layers.MaxPooling2D(
        pool_size=(2, 2),
        name=f"{block_name}_pool"
    )(x)

x = layers.GlobalAveragePooling2D(
    name="global_average_pooling"
)(x)
x = layers.Dense(
    256,
    activation="tanh",
    name="dense_features"
)(x)
x = layers.Dropout(
    0.5
)(x)
outputs = layers.Dense(
    NUM_CLASSES,
    activation="softmax",
    name="classification_output"
)(x)

model = models.Model(
    inputs=inputs,
    outputs=outputs,
    name="Pneumonia_VGG_Tanh_CNN"
)


print("MODEL CREATED SUCCESSFULLY")

model.summary()


In [ ]:
optimizer = tf.keras.optimizers.Adam(
    learning_rate=LEARNING_RATE
)

model.compile(
    optimizer=optimizer,

    loss="sparse_categorical_crossentropy",

    metrics=[
        "accuracy"
    ]
)

print("MODEL COMPILATION COMPLETED")

print("Optimizer: Adam")
print("Loss: Sparse Categorical Crossentropy")
print("Activation: Tanh")
print("Output activation: Softmax")
print("Learning rate:", LEARNING_RATE)


In [ ]:
BEST_MODEL_PATH = OUTPUT_DIR / "best_pneumonia_cnn.keras"

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=7,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(BEST_MODEL_PATH),
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=0.000001,
        verbose=1
    ),
    tf.keras.callbacks.CSVLogger(
        str(OUTPUT_DIR / "training_history.csv")
    )
]

print("TRAINING CALLBACKS CONFIGURED")


In [ ]:
history_df = pd.DataFrame(history.history)
plt.figure(figsize=(9, 5))

plt.plot(
    history_df["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history_df["val_accuracy"],
    label="Validation Accuracy"
)

plt.title("CNN Training and Validation Accuracy")

plt.xlabel("Epoch")

plt.ylabel("Accuracy")

plt.legend()

plt.grid(alpha=0.3)

plt.show()
plt.figure(figsize=(9, 5))

plt.plot(
    history_df["loss"],
    label="Training Loss"
)

plt.plot(
    history_df["val_loss"],
    label="Validation Loss"
)

plt.title("CNN Training and Validation Loss")

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.legend()

plt.grid(alpha=0.3)

plt.show()
history_df.to_csv(
    OUTPUT_DIR / "training_history_full.csv",
    index=False
)

print("TRAINING GRAPHS GENERATED")

print("\nEPOCH RESULTS:")

print(history_df.tail())


In [ ]:
#final model evaluation after each and everything is completed
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
model = tf.keras.models.load_model(
    BEST_MODEL_PATH
)
test_loss, test_accuracy = model.evaluate(
    test_ds,
    verbose=1
)
y_true = test_df["label_id"].values

y_prob = model.predict(
    test_ds,
    verbose=1
)

y_pred = np.argmax(
    y_prob,
    axis=1
)
accuracy = accuracy_score(
    y_true,
    y_pred
)

precision = precision_score(
    y_true,
    y_pred,
    average="macro",
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    average="macro",
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    average="macro",
    zero_division=0
)


print("\nFINAL TEST RESULTS")

print("=" * 45)

print("Test Loss:", round(test_loss, 4))

print("Test Accuracy:", round(accuracy, 4))

print("Macro Precision:", round(precision, 4))

print("Macro Recall:", round(recall, 4))

print("Macro F1-score:", round(f1, 4))
print("\nCLASSIFICATION REPORT")

report = classification_report(
    y_true,
    y_pred,
    target_names=CLASS_NAMES,
    zero_division=0
)
print(report)
# Report
report_dict = classification_report(
    y_true,
    y_pred,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)

pd.DataFrame(report_dict).transpose().to_csv(
    OUTPUT_DIR / "classification_report.csv"
)


In [ ]:
#this cell is completey for generating the convalusion matrix
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1, 2]
)

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES
)

plt.title("CNN Confusion Matrix")

plt.xlabel("Predicted Class")

plt.ylabel("Actual Class")

plt.tight_layout()

plt.show()
cm_normalized = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1, 2],
    normalize="true"
)

plt.figure(figsize=(8, 6))

sns.heatmap(
    cm_normalized,
    annot=True,
    fmt=".2f",
    cmap="Greens",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES
)

plt.title("Normalized Confusion Matrix")

plt.xlabel("Predicted Class")

plt.ylabel("Actual Class")

plt.tight_layout()

plt.show()
pd.DataFrame(
    cm,
    index=CLASS_NAMES,
    columns=CLASS_NAMES
).to_csv(
    OUTPUT_DIR / "confusion_matrix.csv"
)

print("CONFUSION MATRIX SAVED")


In [ ]:
#generating the output of the results
prediction_df = test_df[
    ["image_path", "filename", "label"]
].copy()

prediction_df["actual_class"] = [
    CLASS_NAMES[i] for i in y_true
]

prediction_df["predicted_class"] = [
    CLASS_NAMES[i] for i in y_pred
]

prediction_df["confidence"] = np.max(
    y_prob,
    axis=1
)

prediction_df["correct"] = (
    prediction_df["actual_class"]
    == prediction_df["predicted_class"]
)
print("INDIVIDUAL TEST PREDICTIONS")

print(
    prediction_df[
        [
            "filename",
            "actual_class",
            "predicted_class",
            "confidence",
            "correct"
        ]
    ].head(20).to_string(index=False)
)
prediction_df.to_csv(
    OUTPUT_DIR / "test_predictions.csv",
    index=False
)

print("\nCorrect predictions:",
      prediction_df["correct"].sum())

print("Incorrect predictions:",
      (~prediction_df["correct"]).sum())

print("Total predictions:", len(prediction_df))


In [ ]:
#introduction of gradcam to the model
import cv2

def generate_gradcam(
    model,
    image_tensor,
    last_conv_layer_name,
    class_index=None
):

    # Create model for Grad-CAM
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[
            model.get_layer(
                last_conv_layer_name
            ).output,
            model.output
        ]
    )

    # Calculate gradients
    with tf.GradientTape() as tape:

        conv_outputs, predictions = grad_model(
            image_tensor,
            training=False
        )

        if class_index is None:

            class_index = tf.argmax(
                predictions[0]
            )

        class_score = predictions[
            :, class_index
        ]

    gradients = tape.gradient(
        class_score,
        conv_outputs
    )

    # Average gradients
    pooled_gradients = tf.reduce_mean(
        gradients,
        axis=(0, 1, 2)
    )

    # Extract feature maps
    feature_maps = conv_outputs[0]

    # Weighted feature maps
    heatmap = tf.reduce_sum(
        feature_maps * pooled_gradients,
        axis=-1
    )

    # Apply ReLU
    heatmap = tf.maximum(
        heatmap,
        0
    )

    # Normalize heatmap
    heatmap = heatmap / (
        tf.reduce_max(heatmap) + 1e-8
    )

    return heatmap.numpy(), predictions.numpy()


# Select one test image
sample = test_df.sample(
    1,
    random_state=SEED
).iloc[0]

# Load and preprocess image
image = Image.open(
    sample["image_path"]
).convert("L")

original_image = np.array(
    image.resize((IMG_SIZE, IMG_SIZE))
)

image_tensor, _ = preprocess_image(
    tf.constant(sample["image_path"]),
    tf.constant(sample["label_id"])
)

image_tensor = tf.expand_dims(
    image_tensor,
    axis=0
)


# Generate heatmap
heatmap, prediction = generate_gradcam(
    model,
    image_tensor,
    "block5_conv2"
)

predicted_class = np.argmax(
    prediction[0]
)

# Resize heatmap
heatmap = cv2.resize(
    heatmap,
    (IMG_SIZE, IMG_SIZE)
)

heatmap = np.uint8(
    255 * heatmap
)

# Apply color map
heatmap_color = cv2.applyColorMap(
    heatmap,
    cv2.COLORMAP_JET
)

heatmap_color = cv2.cvtColor(
    heatmap_color,
    cv2.COLOR_BGR2RGB
)

# Convert original image to RGB
base_image = cv2.cvtColor(
    original_image,
    cv2.COLOR_GRAY2RGB
)

# Overlay Grad-CAM
overlay = cv2.addWeighted(
    base_image,
    0.6,
    heatmap_color,
    0.4,
    0
)
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)

plt.imshow(original_image, cmap="gray")

plt.title("Original X-Ray")

plt.axis("off")


plt.subplot(1, 3, 2)

plt.imshow(heatmap, cmap="jet")

plt.title("Grad-CAM Heatmap")

plt.axis("off")


plt.subplot(1, 3, 3)

plt.imshow(overlay)

plt.title("Grad-CAM Overlay")

plt.axis("off")


plt.tight_layout()

plt.show()


print("ACTUAL CLASS:", sample["label"])

print(
    "PREDICTED CLASS:",
    CLASS_NAMES[predicted_class]
)

print(
    "PREDICTION CONFIDENCE:",
    round(float(np.max(prediction)), 4)
)
